# VLM-Gated DreamerV3 on Atari Boxing — FINAL RUN

**Bulletproof overnight run.** Click Run All and go to sleep.

## What this notebook does:
1. ✅ Tests **multiple VLM models** (Gemini + HuggingFace fallback)
2. ✅ Auto-stops at **350K steps OR 5 hours** (whichever first)
3. ✅ Streams every episode + VLM call to disk (crash-safe)
4. ✅ Adaptive threshold + guaranteed periodic VLM calls
5. ✅ Hard cap of **20 VLM calls** total (API safety)
6. ✅ **All analysis cells run** even if training crashes
7. ✅ Compares against your existing baseline (787K vanilla run)

## Configuration (locked):
- **Game:** Atari Boxing
- **Steps:** 350,000 max
- **Time:** 5 hours max  
- **VLM calls:** 8-15 expected, 20 hard cap
- **VLM models:** Gemini 2.5 Flash → 2.0 Flash → 1.5 Flash → HF LLaVA → heuristic fallback

---

In [1]:
# Cell 1 — Environment check
import os, sys

# Detect platform (Kaggle vs Colab vs local)
if os.path.exists('/kaggle/working'):
    PLATFORM = 'kaggle'
    WORKDIR = '/kaggle/working'
elif os.path.exists('/content'):
    PLATFORM = 'colab'
    WORKDIR = '/content'
else:
    PLATFORM = 'local'
    WORKDIR = os.path.expanduser('~/vlm_run')
    os.makedirs(WORKDIR, exist_ok=True)

os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)

print(f"Platform: {PLATFORM}")
print(f"Workdir:  {WORKDIR}")
print(f"Python:   {sys.version.split()[0]}")

# Check GPU
try:
    import jax
    devs = jax.devices()
    print(f"JAX devices: {devs}")
    assert any('cuda' in str(d).lower() or 'gpu' in str(d).lower() for d in devs), "No GPU!"
    print("[OK] GPU detected via JAX")
except Exception as e:
    print(f"[WARN] JAX check: {e}")
    # Try torch as fallback check
    try:
        import torch
        print(f"PyTorch CUDA: {torch.cuda.is_available()}")
    except Exception:
        print("[WARN] Couldn't verify GPU - will continue anyway")

print("\n[OK] Environment ready.")

Platform: colab
Workdir:  /content
Python:   3.12.13
JAX devices: [CudaDevice(id=0)]
[OK] GPU detected via JAX

[OK] Environment ready.


In [2]:
# Cell 2 — API KEYS (already filled in, no input needed)
# These are the user's keys, embedded for one-click overnight run

GEMINI_API_KEY = "YOUR_GEMINI_API_KEY_HERE"
HF_TOKEN = "YOUR_HF_API_KEY_HERE"

# Save to env vars for subprocess access
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["HF_TOKEN"] = HF_TOKEN

print(f"Gemini key:  {'*' * 20}{GEMINI_API_KEY[-6:]}")
print(f"HF token:    {'*' * 20}{HF_TOKEN[-6:]}")
print("[OK] Keys loaded into environment.")

Gemini key:  ********************KqgalA
HF token:    ********************yAYAle
[OK] Keys loaded into environment.


In [3]:
# Cell 3 — Install dependencies (silent, fast)
import subprocess, sys

def pip_install(pkgs, quiet=True):
    cmd = [sys.executable, '-m', 'pip', 'install', '--no-cache-dir']
    if quiet: cmd.append('-q')
    cmd.extend(pkgs)
    try:
        subprocess.run(cmd, check=True, capture_output=True)
        return True
    except subprocess.CalledProcessError as e:
        print(f"[WARN] Install failed: {e.stderr.decode()[:200] if e.stderr else 'unknown'}")
        return False

# Core packages
pip_install(['google-generativeai', 'Pillow', 'requests'])
print("[OK] Gemini SDK + Pillow + requests installed")

# HuggingFace (for backup VLM)
pip_install(['huggingface_hub'])
print("[OK] HuggingFace hub installed")

[OK] Gemini SDK + Pillow + requests installed
[OK] HuggingFace hub installed


In [4]:
# Cell 4 — Clone DreamerV3
import os, subprocess

REPO_DIR = os.path.join(WORKDIR, 'dreamerv3')
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/danijar/dreamerv3.git', REPO_DIR], check=True)
    print('[OK] Cloned dreamerv3')
else:
    print('[OK] dreamerv3 already exists')

# Install dreamerv3 requirements
os.chdir(REPO_DIR)
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q', '-r', 'requirements.txt'],
                       capture_output=True)
if result.returncode == 0:
    print('[OK] DreamerV3 requirements installed')
else:
    print(f'[WARN] Some requirements failed: {result.stderr.decode()[:200]}')

# Verify import
try:
    import elements
    print('[OK] elements module imports successfully')
except ImportError as e:
    print(f'[ERROR] elements import failed: {e}')

os.chdir(WORKDIR)
print(f"CWD: {os.getcwd()}")

[OK] Cloned dreamerv3
[OK] DreamerV3 requirements installed
[OK] elements module imports successfully
CWD: /content


In [5]:
# Cell 5 — PRE-FLIGHT CHECK: Test all VLM models
# This is the critical "fail-fast" cell. If something is broken, we know NOW
# instead of after 5 hours of training.

import google.generativeai as genai
import requests
import io
import numpy as np
from PIL import Image
import time

genai.configure(api_key=GEMINI_API_KEY)

# Test frame: simple gradient as fake Boxing screenshot
def make_test_frame():
    arr = np.zeros((84, 84, 3), dtype=np.uint8)
    arr[:, :, 0] = np.linspace(0, 255, 84).astype(np.uint8)[None, :]
    arr[:, :, 1] = np.linspace(0, 255, 84).astype(np.uint8)[:, None]
    return Image.fromarray(arr)

test_frame = make_test_frame()
test_prompt = "Reply with exactly: OK"

# ============================================================
# Test Gemini models in priority order
# ============================================================
GEMINI_MODELS_TO_TRY = [
    "gemini-2.5-flash",
    "gemini-2.0-flash",
    "gemini-1.5-flash",
    "gemini-1.5-flash-8b",
    "gemini-2.5-flash-lite",
]

working_gemini = []
print("=" * 60)
print("TESTING GEMINI MODELS")
print("=" * 60)

for model_name in GEMINI_MODELS_TO_TRY:
    try:
        model = genai.GenerativeModel(model_name)
        # Test text only first (cheaper)
        resp = model.generate_content(test_prompt)
        text = getattr(resp, 'text', '') or ''
        if text:
            # Now test with image
            resp2 = model.generate_content([test_prompt, test_frame])
            text2 = getattr(resp2, 'text', '') or ''
            if text2:
                working_gemini.append(model_name)
                print(f"  [OK]    {model_name:<30} text+image: '{text2[:30].strip()}'")
            else:
                print(f"  [TEXT]  {model_name:<30} text only, no image support")
        else:
            print(f"  [FAIL]  {model_name:<30} empty response")
    except Exception as e:
        err = str(e)[:80]
        print(f"  [FAIL]  {model_name:<30} {err}")
    time.sleep(0.5)  # Rate limit courtesy

# ============================================================
# Test HuggingFace Inference API
# ============================================================
print()
print("=" * 60)
print("TESTING HUGGINGFACE BACKUP")
print("=" * 60)

HF_MODELS_TO_TRY = [
    "Salesforce/blip-image-captioning-large",
    "Salesforce/blip-image-captioning-base",
]

working_hf = []
for hf_model in HF_MODELS_TO_TRY:
    try:
        # Send image as bytes
        buf = io.BytesIO()
        test_frame.save(buf, format='PNG')
        buf.seek(0)

        url = f"https://api-inference.huggingface.co/models/{hf_model}"
        headers = {"Authorization": f"Bearer {HF_TOKEN}"}

        r = requests.post(url, headers=headers, data=buf.getvalue(), timeout=30)
        if r.status_code == 200:
            result = r.json()
            working_hf.append(hf_model)
            preview = str(result)[:80]
            print(f"  [OK]    {hf_model:<45} -> {preview}")
        elif r.status_code == 503:
            # Model loading - count as working (will be ready when needed)
            working_hf.append(hf_model)
            print(f"  [LOAD]  {hf_model:<45} loading (will retry later)")
        else:
            print(f"  [FAIL]  {hf_model:<45} HTTP {r.status_code}: {r.text[:80]}")
    except Exception as e:
        print(f"  [FAIL]  {hf_model:<45} {str(e)[:80]}")

# ============================================================
# Final verdict
# ============================================================
print()
print("=" * 60)
print("PRE-FLIGHT SUMMARY")
print("=" * 60)
print(f"  Working Gemini models: {len(working_gemini)} / {len(GEMINI_MODELS_TO_TRY)}")
for m in working_gemini: print(f"    - {m}")
print(f"  Working HF models:     {len(working_hf)} / {len(HF_MODELS_TO_TRY)}")
for m in working_hf: print(f"    - {m}")

# Save list for training cell
TOTAL_VLM_OPTIONS = len(working_gemini) + len(working_hf)
PRIMARY_GEMINI = working_gemini[0] if working_gemini else None
PRIMARY_HF = working_hf[0] if working_hf else None

if TOTAL_VLM_OPTIONS == 0:
    print()
    print("[CRITICAL] NO VLM models work! Training will fall back to heuristic.")
    print("           Consider checking your API keys.")
elif TOTAL_VLM_OPTIONS >= 2:
    print()
    print(f"[OK] {TOTAL_VLM_OPTIONS} VLM options available. Excellent redundancy.")
else:
    print()
    print(f"[OK] {TOTAL_VLM_OPTIONS} VLM option available. Continuing with caution.")

print()
print(f"  Primary Gemini: {PRIMARY_GEMINI}")
print(f"  Primary HF:     {PRIMARY_HF}")
print()
print("[OK] Pre-flight complete. Training will proceed regardless of VLM status.")
print("     (Heuristic fallback ensures training never crashes from VLM issues.)")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


TESTING GEMINI MODELS
  [OK]    gemini-2.5-flash               text+image: 'OK'


  [FAIL]  gemini-2.0-flash               429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flas


  [FAIL]  gemini-1.5-flash               404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flas


  [FAIL]  gemini-1.5-flash-8b            404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flas
  [OK]    gemini-2.5-flash-lite          text+image: 'OK'

TESTING HUGGINGFACE BACKUP
  [FAIL]  Salesforce/blip-image-captioning-large        HTTP 404: <!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Error</tit
  [FAIL]  Salesforce/blip-image-captioning-base         HTTP 404: <!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Error</tit

PRE-FLIGHT SUMMARY
  Working Gemini models: 2 / 5
    - gemini-2.5-flash
    - gemini-2.5-flash-lite
  Working HF models:     0 / 2

[OK] 2 VLM options available. Excellent redundancy.

  Primary Gemini: gemini-2.5-flash
  Primary HF:     None

[OK] Pre-flight complete. Training will proceed regardless of VLM status.
     (Heuristic fallback ensures training never crashes from VLM issues.)


In [6]:
# Cell 6 — Write IMPROVED vlm_gate.py
# Multi-tier VLM with adaptive threshold + guaranteed calls + hard caps

vlm_gate_code = """
import os, json, time, io, base64
import numpy as np
from collections import deque

# Try imports (none are required - all have fallbacks)
try:
    import google.generativeai as genai
    _HAS_GEMINI = True
except Exception:
    _HAS_GEMINI = False

try:
    import requests
    _HAS_REQUESTS = True
except Exception:
    _HAS_REQUESTS = False

try:
    from PIL import Image
    _HAS_PIL = True
except Exception:
    _HAS_PIL = False


# Boxing action names
ACTION_NAMES = {
    0: 'NOOP', 1: 'FIRE', 2: 'UP', 3: 'RIGHT', 4: 'LEFT', 5: 'DOWN',
    6: 'UPRIGHT', 7: 'UPLEFT', 8: 'DOWNRIGHT', 9: 'DOWNLEFT',
    10: 'UPFIRE', 11: 'RIGHTFIRE', 12: 'LEFTFIRE', 13: 'DOWNFIRE',
    14: 'UPRIGHTFIRE', 15: 'UPLEFTFIRE', 16: 'DOWNRIGHTFIRE', 17: 'DOWNLEFTFIRE'
}


class VLMGate:
    \"\"\"
    Multi-tier VLM gate with adaptive thresholding and hard caps.

    Trigger paths:
      A) Adaptive surprise threshold (85th percentile of recent window)
      B) Guaranteed periodic call (every GUARANTEED_INTERVAL steps)

    Hard limits:
      - Max calls per hour (API safety)
      - Max total calls (experiment budget)
      - Max calls per 100K steps (per-phase cap)

    VLM model fallback chain:
      1. Primary Gemini model
      2. Secondary Gemini models
      3. HuggingFace BLIP (image captioning)
      4. Heuristic fallback (always works, never crashes)
    \"\"\"

    def __init__(self,
                 gemini_api_key='',
                 hf_token='',
                 gemini_models=None,      # List of Gemini model names to try
                 hf_models=None,          # List of HF model names to try
                 # Adaptive threshold
                 threshold_window=5000,
                 threshold_percentile=85,
                 threshold_min=1.2,
                 threshold_max=2.5,
                 # Queue
                 queue_len=4,
                 # Cooldown (adaptive based on phase)
                 cooldown_early=30,
                 cooldown_mid=50,
                 cooldown_late=70,
                 # Guaranteed calls
                 guaranteed_interval=50000,
                 # Hard caps
                 max_calls_total=20,
                 max_calls_per_hour=8,
                 max_calls_per_100k=6,
                 # Reward shaping
                 reward_scale=0.25,
                 # Warmup
                 warmup_steps=5000,
                 # Logging
                 log_path='/tmp/vlm_calls.jsonl',
                 enabled=True):

        # Config
        self.gemini_api_key = gemini_api_key
        self.hf_token = hf_token
        self.gemini_models = gemini_models or ['gemini-2.5-flash', 'gemini-2.0-flash', 'gemini-1.5-flash']
        self.hf_models = hf_models or ['Salesforce/blip-image-captioning-large']

        self.threshold_window = threshold_window
        self.threshold_percentile = threshold_percentile
        self.threshold_min = threshold_min
        self.threshold_max = threshold_max

        self.queue_len = queue_len
        self.cooldown_early = cooldown_early
        self.cooldown_mid = cooldown_mid
        self.cooldown_late = cooldown_late
        self.guaranteed_interval = guaranteed_interval

        self.max_calls_total = max_calls_total
        self.max_calls_per_hour = max_calls_per_hour
        self.max_calls_per_100k = max_calls_per_100k

        self.reward_scale = reward_scale
        self.warmup_steps = warmup_steps
        self.log_path = log_path
        self.enabled = enabled

        # State
        self.total_steps = 0
        self.total_calls = 0
        self.total_triggers = 0
        self.last_call_step = -guaranteed_interval  # Allow first call after warmup
        self._cooldown = 0

        self.surprise_history = deque(maxlen=threshold_window)
        self.recent_rewards = deque(maxlen=queue_len)
        self.call_timestamps = deque(maxlen=200)  # For per-hour rate limiting
        self.calls_in_current_100k_window = 0
        self.current_100k_window = 0  # Which 100k window we're in

        # Cached suggestion (last VLM response)
        self.cached_suggestion = None
        self.cached_until_step = 0

        # Configure Gemini once
        if _HAS_GEMINI and self.gemini_api_key:
            try:
                genai.configure(api_key=self.gemini_api_key)
            except Exception:
                pass

        # Open log file (JSONL streaming)
        try:
            os.makedirs(os.path.dirname(self.log_path), exist_ok=True)
        except Exception:
            pass
        # Touch the file
        with open(self.log_path, 'a'): pass

        print(f'[VLMGate] Initialized. Enabled={enabled}, models={len(self.gemini_models)} gemini + {len(self.hf_models)} hf')
        print(f'[VLMGate] Adaptive: percentile={threshold_percentile}, range=[{threshold_min}, {threshold_max}]')
        print(f'[VLMGate] Guaranteed: every {guaranteed_interval} steps')
        print(f'[VLMGate] Caps: {max_calls_total} total, {max_calls_per_hour}/hr, {max_calls_per_100k}/100k')

    def _get_cooldown(self):
        \"\"\"Adaptive cooldown based on training phase.\"\"\"
        if self.total_steps < 100000:
            return self.cooldown_early
        elif self.total_steps < 300000:
            return self.cooldown_mid
        else:
            return self.cooldown_late

    def _get_threshold(self):
        \"\"\"Adaptive threshold from recent surprise distribution.\"\"\"
        if len(self.surprise_history) < 100:
            return self.threshold_min  # Permissive during early training

        abs_surprises = [abs(s) for s in self.surprise_history]
        try:
            pct = np.percentile(abs_surprises, self.threshold_percentile)
            return float(np.clip(pct, self.threshold_min, self.threshold_max))
        except Exception:
            return self.threshold_min

    def _check_caps(self):
        \"\"\"Return True if we're under all caps and can call VLM.\"\"\"
        # Total cap
        if self.total_calls >= self.max_calls_total:
            return False, 'total_cap'

        # Per-hour cap
        now = time.time()
        # Remove timestamps older than 1 hour
        while self.call_timestamps and now - self.call_timestamps[0] > 3600:
            self.call_timestamps.popleft()
        if len(self.call_timestamps) >= self.max_calls_per_hour:
            return False, 'hourly_cap'

        # Per-100k cap
        current_window = self.total_steps // 100000
        if current_window != self.current_100k_window:
            self.current_100k_window = current_window
            self.calls_in_current_100k_window = 0
        if self.calls_in_current_100k_window >= self.max_calls_per_100k:
            return False, '100k_cap'

        return True, 'ok'

    def step(self, frame, reward, done=False):
        \"\"\"
        Called every environment step.
        Returns: (reward_bonus, info_dict)
        \"\"\"
        self.total_steps += 1
        info = {'vlm_called': False, 'reason': None, 'surprise': 0.0}

        if not self.enabled:
            return 0.0, info

        # Update reward queue and surprise tracking
        self.recent_rewards.append(float(reward))
        if len(self.recent_rewards) >= 2:
            mean_r = float(np.mean(list(self.recent_rewards)[:-1])) if len(self.recent_rewards) > 1 else 0.0
            std_r = float(np.std(list(self.recent_rewards)[:-1])) + 1e-6
            surprise = (float(reward) - mean_r) / std_r
            self.surprise_history.append(surprise)
            info['surprise'] = surprise
        else:
            surprise = 0.0

        # Decrement cooldown
        if self._cooldown > 0:
            self._cooldown -= 1

        # Skip during warmup
        if self.total_steps < self.warmup_steps:
            return 0.0, info

        # Check if we should fire VLM
        should_fire = False
        reason = None

        # Path A: Guaranteed periodic call
        steps_since_last = self.total_steps - self.last_call_step
        if steps_since_last >= self.guaranteed_interval:
            should_fire = True
            reason = 'guaranteed'

        # Path B: Adaptive surprise threshold
        elif self._cooldown == 0 and len(self.surprise_history) >= self.queue_len:
            threshold = self._get_threshold()
            recent = list(self.surprise_history)[-self.queue_len:]
            n_above = sum(1 for s in recent if abs(s) >= threshold)
            if n_above >= max(2, self.queue_len // 2):  # Half of queue must exceed
                should_fire = True
                reason = 'adaptive_surprise'
                self.total_triggers += 1

        if not should_fire:
            return 0.0, info

        # Check hard caps before actually calling
        can_call, cap_reason = self._check_caps()
        if not can_call:
            info['reason'] = f'capped_{cap_reason}'
            return 0.0, info

        # FIRE VLM!
        info['vlm_called'] = True
        info['reason'] = reason
        self.total_calls += 1
        self.last_call_step = self.total_steps
        self.call_timestamps.append(time.time())
        self.calls_in_current_100k_window += 1
        self._cooldown = self._get_cooldown()

        # Get VLM suggestion (with full fallback chain)
        suggestion = self._call_vlm_with_fallback(frame, reward, surprise)

        # Apply reward shaping
        bonus = float(suggestion.get('reward_bonus', 0.0)) * self.reward_scale

        # Log call to JSONL (streaming, crash-safe)
        log_entry = {
            'step': self.total_steps,
            'time': time.time(),
            'reason': reason,
            'reward': float(reward),
            'surprise': float(surprise),
            'threshold': self._get_threshold(),
            'cooldown': self._cooldown,
            'model_used': suggestion.get('model_used', 'unknown'),
            'description': suggestion.get('description', '')[:200],
            'actions': suggestion.get('actions', []),
            'reward_bonus': bonus,
            'raw_bonus': float(suggestion.get('reward_bonus', 0.0)),
            'total_calls_so_far': self.total_calls,
        }
        try:
            with open(self.log_path, 'a') as f:
                f.write(json.dumps(log_entry) + '\\n')
        except Exception as e:
            print(f'[VLMGate] Log write failed: {e}')

        # Console output
        print(f'[VLM #{self.total_calls}] step={self.total_steps} reason={reason} '
              f'model={suggestion.get(\"model_used\")} bonus={bonus:+.3f} '
              f'desc=\"{suggestion.get(\"description\", \"\")[:60]}\"')

        return bonus, info

    def _call_vlm_with_fallback(self, frame, reward, surprise):
        \"\"\"Try each VLM model in order. Always returns SOMETHING.\"\"\"
        # Convert frame to PIL Image
        pil_frame = self._frame_to_pil(frame)

        prompt = self._build_prompt(reward, surprise)

        # Try Gemini models
        for model_name in self.gemini_models:
            try:
                result = self._call_gemini(model_name, prompt, pil_frame)
                if result:
                    result['model_used'] = model_name
                    return result
            except Exception as e:
                print(f'[VLM] Gemini {model_name} failed: {str(e)[:60]}')
                continue

        # Try HuggingFace as backup
        for hf_model in self.hf_models:
            try:
                result = self._call_huggingface(hf_model, pil_frame)
                if result:
                    result['model_used'] = f'hf:{hf_model}'
                    return result
            except Exception as e:
                print(f'[VLM] HF {hf_model} failed: {str(e)[:60]}')
                continue

        # ULTIMATE FALLBACK: Heuristic (never crashes)
        print('[VLM] All VLM models failed, using heuristic fallback')
        return self._heuristic_fallback(frame, reward, surprise)

    def _frame_to_pil(self, frame):
        \"\"\"Convert numpy frame to PIL Image.\"\"\"
        if not _HAS_PIL:
            return None
        try:
            arr = np.asarray(frame)
            if arr.dtype != np.uint8:
                arr = (arr * 255).clip(0, 255).astype(np.uint8) if arr.max() <= 1.0 else arr.astype(np.uint8)
            if arr.ndim == 2:
                arr = np.stack([arr]*3, axis=-1)
            elif arr.ndim == 3 and arr.shape[0] in (1, 3, 4) and arr.shape[-1] not in (1, 3, 4):
                arr = np.transpose(arr, (1, 2, 0))
            if arr.shape[-1] == 4:
                arr = arr[..., :3]
            # Upscale small frames (Atari is 84x84) for better VLM understanding
            img = Image.fromarray(arr)
            if img.size[0] < 200:
                img = img.resize((img.size[0]*4, img.size[1]*4), Image.NEAREST)
            return img
        except Exception as e:
            print(f'[VLM] Frame conversion failed: {e}')
            return None

    def _build_prompt(self, reward, surprise):
        action_list = ', '.join(f'{k}={v}' for k, v in list(ACTION_NAMES.items())[:10])
        return f\"\"\"You are watching an Atari Boxing match. The agent (left fighter) is trying to punch the opponent (right fighter).

Recent reward: {reward:.1f}
Surprise level: {surprise:.2f}

Available actions: {action_list}, ...up to 17 (movement+fire combos).

Respond ONLY with valid JSON in this exact format (no markdown, no explanation):
{{
  \"description\": \"<one sentence: what's happening>\",
  \"actions\": [<list of 1-3 best action numbers 0-17>],
  \"reward_bonus\": <float between -1.0 and 1.0, where positive = good situation>
}}\"\"\"

    def _call_gemini(self, model_name, prompt, pil_frame):
        if not _HAS_GEMINI or not self.gemini_api_key:
            return None
        try:
            model = genai.GenerativeModel(model_name)
            content = [prompt]
            if pil_frame is not None:
                content.append(pil_frame)

            resp = model.generate_content(content, request_options={'timeout': 30})
            text = getattr(resp, 'text', '') or ''
            return self._parse_response(text)
        except Exception as e:
            raise e

    def _call_huggingface(self, hf_model, pil_frame):
        if not _HAS_REQUESTS or not self.hf_token or pil_frame is None:
            return None
        try:
            buf = io.BytesIO()
            pil_frame.save(buf, format='PNG')
            buf.seek(0)

            url = f'https://api-inference.huggingface.co/models/{hf_model}'
            headers = {'Authorization': f'Bearer {self.hf_token}'}
            r = requests.post(url, headers=headers, data=buf.getvalue(), timeout=30)

            if r.status_code != 200:
                raise RuntimeError(f'HTTP {r.status_code}')

            result = r.json()
            # BLIP returns [{'generated_text': '...'}]
            if isinstance(result, list) and result:
                description = result[0].get('generated_text', '')[:200]
            else:
                description = str(result)[:200]

            # Convert caption to actions (heuristic - HF models don't pick actions)
            return {
                'description': description,
                'actions': [11, 12],  # RIGHTFIRE, LEFTFIRE - punch actions
                'reward_bonus': 0.0,  # Neutral
            }
        except Exception as e:
            raise e

    def _parse_response(self, text):
        \"\"\"Parse VLM JSON response, with cleanup.\"\"\"
        if not text:
            return None
        try:
            # Strip markdown code fences
            text = text.strip()
            if text.startswith('```'):
                text = text.split('```')[1]
                if text.startswith('json'):
                    text = text[4:]
            text = text.strip()

            # Find JSON object
            start = text.find('{')
            end = text.rfind('}')
            if start >= 0 and end > start:
                text = text[start:end+1]

            result = json.loads(text)

            # Validate
            actions = result.get('actions', [])
            if not isinstance(actions, list):
                actions = [actions] if isinstance(actions, int) else []
            actions = [int(a) for a in actions if isinstance(a, (int, float)) and 0 <= a <= 17][:3]

            bonus = float(result.get('reward_bonus', 0.0))
            bonus = max(-1.0, min(1.0, bonus))

            return {
                'description': str(result.get('description', ''))[:200],
                'actions': actions,
                'reward_bonus': bonus,
            }
        except Exception as e:
            return {
                'description': f'parse_fail: {text[:80]}',
                'actions': [],
                'reward_bonus': 0.0,
            }

    def _heuristic_fallback(self, frame, reward, surprise):
        \"\"\"Pure heuristic - never crashes, no API needed.\"\"\"
        # Simple logic: if recent reward was negative, suggest defensive
        # If positive, suggest aggressive
        if reward > 0:
            actions = [11, 12]  # RIGHTFIRE, LEFTFIRE (punch)
            bonus = 0.1
            desc = 'heuristic: aggressive (recent +reward)'
        elif reward < 0:
            actions = [3, 4]  # RIGHT, LEFT (move)
            bonus = -0.1
            desc = 'heuristic: defensive (recent -reward)'
        else:
            actions = [0]  # NOOP
            bonus = 0.0
            desc = 'heuristic: neutral'

        return {
            'description': desc,
            'actions': actions,
            'reward_bonus': bonus,
            'model_used': 'heuristic',
        }

    def get_stats(self):
        return {
            'total_steps': self.total_steps,
            'total_calls': self.total_calls,
            'total_triggers': self.total_triggers,
            'call_rate_pct': 100.0 * self.total_calls / max(1, self.total_steps),
            'current_threshold': self._get_threshold(),
            'current_cooldown': self._cooldown,
        }
"""

with open(os.path.join(WORKDIR, 'vlm_gate.py'), 'w') as f:
    f.write(vlm_gate_code)

print(f"[OK] Wrote vlm_gate.py ({len(vlm_gate_code)} bytes)")

# Quick syntax check
import py_compile
try:
    py_compile.compile(os.path.join(WORKDIR, 'vlm_gate.py'), doraise=True)
    print("[OK] vlm_gate.py syntax valid")
except py_compile.PyCompileError as e:
    print(f"[ERROR] Syntax error: {e}")

[OK] Wrote vlm_gate.py (18389 bytes)
[OK] vlm_gate.py syntax valid


In [7]:
# Cell 7 — Write vlm_gym_wrapper.py
# Thin wrapper that hooks VLMGate into the env step

wrapper_code = """
import sys, os
sys.path.insert(0, os.environ.get('VLM_WORKDIR', '/kaggle/working'))
import numpy as np
from vlm_gate import VLMGate


class VLMGymWrapper:
    \"\"\"Wraps a Gym/Atari env to inject VLM-based reward shaping.\"\"\"

    def __init__(self, env,
                 gemini_api_key='',
                 hf_token='',
                 gemini_models=None,
                 hf_models=None,
                 enabled=True,
                 log_path='/tmp/vlm_calls.jsonl',
                 reward_scale=0.25,
                 **gate_kwargs):

        self.env = env
        self.gate = VLMGate(
            gemini_api_key=gemini_api_key,
            hf_token=hf_token,
            gemini_models=gemini_models,
            hf_models=hf_models,
            enabled=enabled,
            log_path=log_path,
            reward_scale=reward_scale,
            **gate_kwargs
        )
        # Forward attributes
        self.observation_space = getattr(env, 'observation_space', None)
        self.action_space = getattr(env, 'action_space', None)

    def __getattr__(self, name):
        return getattr(self.env, name)

    def reset(self, *args, **kwargs):
        return self.env.reset(*args, **kwargs)

    def step(self, action):
        result = self.env.step(action)
        # Handle 4 vs 5-tuple gym API
        if len(result) == 4:
            obs, reward, done, info = result
            terminated, truncated = done, False
        else:
            obs, reward, terminated, truncated, info = result
            done = terminated or truncated

        # Get frame for VLM (use observation if it looks like an image)
        frame = obs
        if hasattr(self.env, 'render'):
            try:
                frame = self.env.render()
            except Exception:
                frame = obs

        # Apply VLM gating
        try:
            bonus, vlm_info = self.gate.step(frame, reward, done)
            shaped_reward = reward + bonus
            if isinstance(info, dict):
                info['vlm_bonus'] = bonus
                info['vlm_info'] = vlm_info
        except Exception as e:
            print(f'[VLMGymWrapper] Gate error (continuing): {e}')
            shaped_reward = reward

        if len(result) == 4:
            return obs, shaped_reward, done, info
        else:
            return obs, shaped_reward, terminated, truncated, info
"""

with open(os.path.join(WORKDIR, 'vlm_gym_wrapper.py'), 'w') as f:
    f.write(wrapper_code)

print(f"[OK] Wrote vlm_gym_wrapper.py")

import py_compile
try:
    py_compile.compile(os.path.join(WORKDIR, 'vlm_gym_wrapper.py'), doraise=True)
    print("[OK] vlm_gym_wrapper.py syntax valid")
except py_compile.PyCompileError as e:
    print(f"[ERROR] Syntax error: {e}")

[OK] Wrote vlm_gym_wrapper.py
[OK] vlm_gym_wrapper.py syntax valid


In [8]:
# Cell 8 — Patch DreamerV3 atari.py: inject VLM Gate INTO Atari.step()
#
# CRITICAL FIX: DreamerV3's Atari class uses ALE directly (not Gym), so we cannot
# wrap it with VLMGymWrapper or monkey-patch gym.make. Instead, we directly modify
# the Atari class:
#   1. Add VLM gate init at end of __init__
#   2. Modify step() to call gate after computing reward, before returning obs
import os, glob, re

# Find atari.py
candidates = glob.glob(os.path.join(REPO_DIR, '**', 'atari.py'), recursive=True)
atari_path = None
for c in candidates:
    if 'embodied' in c and 'envs' in c:
        atari_path = c
        break
if atari_path is None and candidates:
    atari_path = candidates[0]

assert atari_path, 'atari.py not found in dreamerv3!'
print(f'Found atari.py: {atari_path}')

# Always start fresh from a backup if we have one (to avoid accumulating bad patches)
backup_path = atari_path + '.original_backup'
if not os.path.exists(backup_path):
    # First run - save the original
    with open(atari_path, 'r') as f:
        orig = f.read()
    # Only save if it doesn't already contain our markers (otherwise it's already patched)
    if 'VLM_GATE_BOXING_PATCHED' not in orig:
        with open(backup_path, 'w') as f:
            f.write(orig)
        print(f'[OK] Saved original atari.py to backup')
    else:
        # Already patched - try to find a backup elsewhere or fail gracefully
        print('[WARN] atari.py already patched but no backup. Re-cloning fresh.')
        import subprocess
        subprocess.run(['rm', '-rf', REPO_DIR], check=False)
        subprocess.run(['git', 'clone', '--depth', '1',
                       'https://github.com/danijar/dreamerv3.git', REPO_DIR], check=True)
        with open(atari_path, 'r') as f:
            orig = f.read()
        with open(backup_path, 'w') as f:
            f.write(orig)
        print('[OK] Fresh clone, saved backup')

# Always start from the clean backup
with open(backup_path, 'r') as f:
    original = f.read()

# Reliable anchor strings (verified to exist in current dreamerv3 master)
INIT_ANCHOR = "    self.done = True"  # Last line of __init__
STEP_ANCHOR = "    obs = self._obs(reward, is_last=last, is_terminal=terminal)"  # Just before return in step()

assert INIT_ANCHOR in original, f"INIT_ANCHOR not found in atari.py - dreamerv3 version may have changed"
assert STEP_ANCHOR in original, f"STEP_ANCHOR not found in atari.py - dreamerv3 version may have changed"

# Build patched content
patched = original

# 1. Add imports at top (after existing imports)
import_block = """
# === VLM_GATE_BOXING_PATCHED_V3 ===
import os as _vlm_os
import sys as _vlm_sys
_VLM_ENABLED = _vlm_os.environ.get('VLM_GATE_ENABLED', '0') == '1'
if _VLM_ENABLED:
    _vlm_workdir = _vlm_os.environ.get('VLM_WORKDIR', '/kaggle/working')
    if _vlm_workdir not in _vlm_sys.path:
        _vlm_sys.path.insert(0, _vlm_workdir)
    try:
        from vlm_gate import VLMGate as _VLMGateClass
        print('[Atari] >>> VLM GATE IMPORT OK <<<')
    except Exception as _e:
        print(f'[Atari] VLM gate import failed: {_e}')
        _VLMGateClass = None
else:
    _VLMGateClass = None
# === END VLM_GATE_IMPORT ===

"""

# Insert import block right before the `class Atari` line
class_anchor = "class Atari("
assert class_anchor in patched, "class Atari not found"
patched = patched.replace(class_anchor, import_block + class_anchor, 1)

# 2. Modify __init__ to create VLM gate (replace INIT_ANCHOR with anchor + gate init)
init_injection = INIT_ANCHOR + """

    # === VLM_GATE_INIT ===
    self._vlm_gate = None
    if _VLM_ENABLED and _VLMGateClass is not None:
        try:
            _gemini_models = _vlm_os.environ.get('VLM_GEMINI_MODELS', '')
            _gemini_models = [m.strip() for m in _gemini_models.split(',') if m.strip()] or None
            _hf_models = _vlm_os.environ.get('VLM_HF_MODELS', '')
            _hf_models = [m.strip() for m in _hf_models.split(',') if m.strip()] or None
            self._vlm_gate = _VLMGateClass(
                gemini_api_key=_vlm_os.environ.get('GEMINI_API_KEY', ''),
                hf_token=_vlm_os.environ.get('HF_TOKEN', ''),
                gemini_models=_gemini_models,
                hf_models=_hf_models,
                enabled=True,
                log_path=_vlm_os.environ.get('VLM_LOG_PATH', '/tmp/vlm_calls.jsonl'),
                reward_scale=float(_vlm_os.environ.get('VLM_REWARD_SCALE', '0.25')),
                threshold_window=int(_vlm_os.environ.get('VLM_THRESHOLD_WINDOW', '5000')),
                threshold_percentile=int(_vlm_os.environ.get('VLM_THRESHOLD_PERCENTILE', '85')),
                threshold_min=float(_vlm_os.environ.get('VLM_THRESHOLD_MIN', '1.2')),
                threshold_max=float(_vlm_os.environ.get('VLM_THRESHOLD_MAX', '2.5')),
                queue_len=int(_vlm_os.environ.get('VLM_QUEUE_LEN', '4')),
                cooldown_early=int(_vlm_os.environ.get('VLM_COOLDOWN_EARLY', '30')),
                cooldown_mid=int(_vlm_os.environ.get('VLM_COOLDOWN_MID', '50')),
                cooldown_late=int(_vlm_os.environ.get('VLM_COOLDOWN_LATE', '70')),
                guaranteed_interval=int(_vlm_os.environ.get('VLM_GUARANTEED_INTERVAL', '50000')),
                max_calls_total=int(_vlm_os.environ.get('VLM_MAX_CALLS_TOTAL', '20')),
                max_calls_per_hour=int(_vlm_os.environ.get('VLM_MAX_CALLS_PER_HOUR', '8')),
                max_calls_per_100k=int(_vlm_os.environ.get('VLM_MAX_CALLS_PER_100K', '6')),
                warmup_steps=int(_vlm_os.environ.get('VLM_WARMUP_STEPS', '5000')),
            )
            print('[Atari] >>> VLM GATE ATTACHED to Atari instance <<<')
        except Exception as _e:
            print(f'[Atari] VLM gate init failed: {_e}')
            import traceback; traceback.print_exc()
    # === END VLM_GATE_INIT ==="""

patched = patched.replace(INIT_ANCHOR, init_injection, 1)

# 3. Modify step() to call VLM gate before returning observation
# Replace the obs assignment line with: gate call + modified reward + obs
step_injection = """    # === VLM_GATE_STEP ===
    if self._vlm_gate is not None:
        try:
            # Get current frame from buffers (most recent)
            _vlm_frame = self.buffers[0] if len(self.buffers) > 0 else None
            _vlm_bonus, _vlm_info = self._vlm_gate.step(_vlm_frame, reward, last)
            reward = float(reward) + float(_vlm_bonus)
        except Exception as _e:
            pass  # Never let VLM crash training
    # === END VLM_GATE_STEP ===
""" + STEP_ANCHOR

patched = patched.replace(STEP_ANCHOR, step_injection, 1)

# Write patched file
with open(atari_path, 'w') as f:
    f.write(patched)

# Verify
with open(atari_path, 'r') as f:
    final = f.read()

# Sanity checks
checks = [
    ('VLM_GATE_BOXING_PATCHED_V3', final),
    ('VLM_GATE_INIT', final),
    ('VLM_GATE_STEP', final),
    ('self._vlm_gate', final),
]
all_ok = True
for marker, content in checks:
    found = marker in content
    print(f"  {'[OK]' if found else '[FAIL]'} marker: {marker}")
    if not found:
        all_ok = False

# Compile-check the patched file
import py_compile
try:
    py_compile.compile(atari_path, doraise=True)
    print(f"[OK] Patched atari.py compiles successfully ({len(final)} bytes)")
except py_compile.PyCompileError as e:
    print(f"[ERROR] Patched atari.py has syntax error: {e}")
    print("[INFO] Restoring original from backup")
    with open(backup_path, 'r') as f:
        orig = f.read()
    with open(atari_path, 'w') as f:
        f.write(orig)
    raise

assert all_ok, "Some markers missing - patch may be incomplete"
print("[OK] atari.py patched successfully with VLM gate")


Found atari.py: /content/dreamerv3/embodied/envs/atari.py
[OK] Saved original atari.py to backup
  [OK] marker: VLM_GATE_BOXING_PATCHED_V3
  [OK] marker: VLM_GATE_INIT
  [OK] marker: VLM_GATE_STEP
  [OK] marker: self._vlm_gate
[OK] Patched atari.py compiles successfully (9058 bytes)
[OK] atari.py patched successfully with VLM gate


In [9]:
# Cell 9 — VLM smoke test with actual VLMGate
# Tests the full pipeline before training starts

import sys, os, numpy as np
os.environ['VLM_WORKDIR'] = WORKDIR
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)

# Force reimport
for mod in list(sys.modules.keys()):
    if mod in ('vlm_gate', 'vlm_gym_wrapper'):
        del sys.modules[mod]

from vlm_gate import VLMGate

# Use the working models from pre-flight
gemini_models = working_gemini if working_gemini else ['gemini-1.5-flash']
hf_models = working_hf if working_hf else []

print(f"Testing with Gemini: {gemini_models[:2]}")
print(f"Testing with HF: {hf_models[:1]}")
print()

# Create gate with low warmup so we can test immediately
gate = VLMGate(
    gemini_api_key=GEMINI_API_KEY,
    hf_token=HF_TOKEN,
    gemini_models=gemini_models,
    hf_models=hf_models,
    threshold_window=100,
    threshold_min=0.5,  # Very low for testing
    threshold_max=2.0,
    queue_len=3,
    cooldown_early=2,
    cooldown_mid=2,
    cooldown_late=2,
    guaranteed_interval=10,  # Force call every 10 steps for test
    max_calls_total=3,  # Small cap for test
    max_calls_per_hour=10,
    max_calls_per_100k=10,
    warmup_steps=2,  # Short warmup
    log_path=os.path.join(WORKDIR, 'vlm_smoke_test.jsonl'),
    enabled=True
)

print("Running 25-step smoke test (should trigger ~2 VLM calls)...")
print("-" * 60)

n_calls = 0
for i in range(25):
    # Fake Boxing frame (gradient pattern)
    frame = np.zeros((84, 84, 3), dtype=np.uint8)
    frame[:, :, 0] = i * 10 % 256
    frame[:, :, 1] = (i * 7 + 50) % 256

    # Reward varies to create surprises
    reward = float(np.random.choice([-1, 0, 0, 0, 1], p=[0.1, 0.3, 0.3, 0.2, 0.1]))

    bonus, info = gate.step(frame, reward)
    if info['vlm_called']:
        n_calls += 1

print("-" * 60)
print()
stats = gate.get_stats()
print(f"Smoke test results:")
print(f"  Total steps:    {stats['total_steps']}")
print(f"  Total calls:    {stats['total_calls']}")
print(f"  Total triggers: {stats['total_triggers']}")
print(f"  Call rate:      {stats['call_rate_pct']:.2f}%")

if n_calls >= 1:
    print(f"\n[OK] Smoke test passed: {n_calls} VLM calls succeeded")
else:
    print(f"\n[WARN] No VLM calls fired - check thresholds (will still work in training)")

print()
print("[OK] Pipeline verified end-to-end. Ready for training.")

Testing with Gemini: ['gemini-2.5-flash', 'gemini-2.5-flash-lite']
Testing with HF: []

[VLMGate] Initialized. Enabled=True, models=2 gemini + 1 hf
[VLMGate] Adaptive: percentile=85, range=[0.5, 2.0]
[VLMGate] Guaranteed: every 10 steps
[VLMGate] Caps: 3 total, 10/hr, 10/100k
Running 25-step smoke test (should trigger ~2 VLM calls)...
------------------------------------------------------------
[VLM #1] step=2 reason=guaranteed model=gemini-2.5-flash bonus=+0.000 desc="The screen is currently dark green with no fighters visible,"
[VLM #2] step=10 reason=adaptive_surprise model=gemini-2.5-flash bonus=-0.225 desc="The screen is completely green, providing no visual informat"
[VLM #3] step=12 reason=adaptive_surprise model=gemini-2.5-flash bonus=+0.000 desc="The screen is completely blank and green, providing no visua"
------------------------------------------------------------

Smoke test results:
  Total steps:    25
  Total calls:    3
  Total triggers: 4
  Call rate:      12.00%

[OK

In [ ]:
# Cell 10 — TRAINING with auto-stop, crash-safe streaming
#
# Stops at: 350K steps OR 5 hours OR exception
# Saves: episodes (every episode), VLM calls (every call), checkpoints (every 25K)
#
# This cell is wrapped in try/finally so the analysis cells WILL run even if
# training crashes mid-way.

import os, sys, subprocess, time, signal, threading, json

# ============================================================
# Configuration
# ============================================================
LOGDIR_VLM = os.path.join(WORKDIR, 'dreamer_boxing_vlm')
VLM_LOG_PATH = os.path.join(WORKDIR, 'vlm_calls.jsonl')

MAX_STEPS = 350_000
MAX_HOURS = 5.0
MAX_SECONDS = MAX_HOURS * 3600

os.makedirs(LOGDIR_VLM, exist_ok=True)

# Clear old VLM log
if os.path.exists(VLM_LOG_PATH):
    os.rename(VLM_LOG_PATH, VLM_LOG_PATH + '.old')
open(VLM_LOG_PATH, 'w').close()

# ============================================================
# Set up training environment
# ============================================================
env = os.environ.copy()
env.pop('XLA_FLAGS', None)
env.pop('TF_XLA_FLAGS', None)
env['JAX_PLATFORMS'] = 'cuda'
env['PYTHONNOUSERSITE'] = '1'
env['VLM_WORKDIR'] = WORKDIR

# Enable VLM
env['VLM_GATE_ENABLED'] = '1'
env['GEMINI_API_KEY'] = GEMINI_API_KEY
env['HF_TOKEN'] = HF_TOKEN
env['VLM_LOG_PATH'] = VLM_LOG_PATH

# VLM models (comma-separated)
env['VLM_GEMINI_MODELS'] = ','.join(gemini_models[:3]) if gemini_models else 'gemini-1.5-flash'
env['VLM_HF_MODELS'] = ','.join(hf_models) if hf_models else ''

# VLM hyperparameters
env['VLM_REWARD_SCALE'] = '0.25'
env['VLM_THRESHOLD_WINDOW'] = '5000'
env['VLM_THRESHOLD_PERCENTILE'] = '85'
env['VLM_THRESHOLD_MIN'] = '1.2'
env['VLM_THRESHOLD_MAX'] = '2.5'
env['VLM_QUEUE_LEN'] = '4'
env['VLM_COOLDOWN_EARLY'] = '30'
env['VLM_COOLDOWN_MID'] = '50'
env['VLM_COOLDOWN_LATE'] = '70'
env['VLM_GUARANTEED_INTERVAL'] = '50000'
env['VLM_MAX_CALLS_TOTAL'] = '20'
env['VLM_MAX_CALLS_PER_HOUR'] = '8'
env['VLM_MAX_CALLS_PER_100K'] = '6'
env['VLM_WARMUP_STEPS'] = '5000'

print("=" * 65)
print("STARTING VLM-GATED TRAINING ON ATARI BOXING")
print("=" * 65)
print(f"Logdir:         {LOGDIR_VLM}")
print(f"VLM log:        {VLM_LOG_PATH}")
print(f"Max steps:      {MAX_STEPS:,}")
print(f"Max time:       {MAX_HOURS} hours")
print(f"Gemini models:  {env['VLM_GEMINI_MODELS']}")
print(f"HF models:      {env['VLM_HF_MODELS'] or '(none)'}")
print(f"Started:        {time.strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 65)
print()

# ============================================================
# Build training command
# ============================================================
train_script = os.path.join(REPO_DIR, 'dreamerv3', 'main.py')
if not os.path.exists(train_script):
    # Try alternative paths
    candidates = [
        os.path.join(REPO_DIR, 'dreamerv3', 'main.py'),
        os.path.join(REPO_DIR, 'main.py'),
        os.path.join(REPO_DIR, 'embodied', 'scripts', 'train.py'),
    ]
    for c in candidates:
        if os.path.exists(c):
            train_script = c
            break

print(f"Training script: {train_script}")

cmd = [
    sys.executable, '-u', train_script,
    '--logdir', LOGDIR_VLM,
    '--configs', 'atari100k', 'size12m',
    '--task', 'atari_boxing',
    '--run.steps', str(MAX_STEPS),
    '--run.log_every', '120',
    '--run.save_every', '900',
]

# ============================================================
# Run training with auto-stop
# ============================================================
training_completed = False
training_error = None
start_time = time.time()

try:
    print("Launching training subprocess...")
    print(f"Command: {' '.join(cmd[:5])} ...")
    print()

    proc = subprocess.Popen(
        cmd,
        env=env,
        cwd=REPO_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    # Stream output with auto-stop
    last_step_seen = 0
    last_progress_time = time.time()

    while True:
        # Check time limit
        elapsed = time.time() - start_time
        if elapsed >= MAX_SECONDS:
            print(f"\n[AUTO-STOP] Time limit reached: {elapsed/3600:.2f} hours")
            proc.terminate()
            try:
                proc.wait(timeout=30)
            except subprocess.TimeoutExpired:
                proc.kill()
            break

        # Check step limit (parsed from stdout)
        if last_step_seen >= MAX_STEPS:
            print(f"\n[AUTO-STOP] Step limit reached: {last_step_seen:,}")
            proc.terminate()
            try:
                proc.wait(timeout=30)
            except subprocess.TimeoutExpired:
                proc.kill()
            break

        # Read line (with timeout via poll)
        line = proc.stdout.readline()
        if not line:
            if proc.poll() is not None:
                # Process ended
                print("\n[INFO] Training process ended naturally")
                training_completed = True
                break
            continue

        # Echo line
        print(line, end='')

        # Parse step number from log lines
        if 'Agent Step' in line:
            try:
                # Extract step from "Agent Step 12_345"
                import re
                m = re.search(r'Agent Step ([\d_]+)', line)
                if m:
                    last_step_seen = int(m.group(1).replace('_', ''))
                    last_progress_time = time.time()
            except Exception:
                pass

        # Stall detection: no progress for 10 minutes
        if time.time() - last_progress_time > 600:
            print(f"\n[STALL] No progress for 10 minutes, stopping")
            proc.terminate()
            try:
                proc.wait(timeout=30)
            except subprocess.TimeoutExpired:
                proc.kill()
            break

    print(f"\n[INFO] Training stopped at step ~{last_step_seen:,}")
    training_completed = True

except KeyboardInterrupt:
    print("\n[INTERRUPT] User stopped training")
    try:
        proc.terminate()
        proc.wait(timeout=10)
    except Exception:
        try: proc.kill()
        except Exception: pass

except Exception as e:
    training_error = str(e)
    print(f"\n[ERROR] Training failed: {e}")
    import traceback
    traceback.print_exc()

finally:
    elapsed = time.time() - start_time
    print()
    print("=" * 65)
    print("TRAINING SESSION ENDED")
    print("=" * 65)
    print(f"Status:    {'COMPLETED' if training_completed else 'CRASHED/STOPPED'}")
    print(f"Elapsed:   {elapsed/3600:.2f} hours ({elapsed/60:.1f} min)")
    print(f"Last step: ~{last_step_seen:,}")
    if training_error:
        print(f"Error:     {training_error}")
    print()
    print(f"Files in logdir:")
    for root, dirs, files in os.walk(LOGDIR_VLM):
        for f in files:
            fp = os.path.join(root, f)
            sz = os.path.getsize(fp) / 1024
            print(f"  {fp.replace(LOGDIR_VLM, '...'):<50s} {sz:>8.1f} KB")

    # Count VLM calls in log
    vlm_call_count = 0
    if os.path.exists(VLM_LOG_PATH):
        with open(VLM_LOG_PATH, 'r') as f:
            vlm_call_count = sum(1 for _ in f)
    print()
    print(f"VLM calls logged: {vlm_call_count}")
    print()
    print("[NEXT] Analysis cells will now run on whatever data exists.")
    print("=" * 65)

STARTING VLM-GATED TRAINING ON ATARI BOXING
Logdir:         /content/dreamer_boxing_vlm
VLM log:        /content/vlm_calls.jsonl
Max steps:      350,000
Max time:       5.0 hours
Gemini models:  gemini-2.5-flash,gemini-2.5-flash-lite
HF models:      (none)
Started:        2026-05-08 21:56:29

Training script: /content/dreamerv3/dreamerv3/main.py
Launching training subprocess...
Command: /usr/bin/python3 -u /content/dreamerv3/dreamerv3/main.py --logdir /content/dreamer_boxing_vlm ...

---  ___                           __   ______ ---
--- |   \ _ _ ___ __ _ _ __  ___ _ \ \ / /__ / ---
--- | |) | '_/ -_) _` | '  \/ -_) '/\ V / |_ \ ---
--- |___/|_| \___\__,_|_|_|_\___|_|  \_/ |___/ ---
Replica: 0 / 1
Logdir: /content/dreamer_boxing_vlm
Run script: train
/content/vlm_gate.py:8: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for mo

In [ ]:
# Cell 11 — Load VLM run metrics (works even on partial data)
import os, json, glob
import numpy as np

LOGDIR_VLM = os.path.join(WORKDIR, 'dreamer_boxing_vlm')

def load_metrics(logdir):
    """Try multiple paths to find metrics.jsonl."""
    candidates = [
        os.path.join(logdir, 'metrics.jsonl'),
        os.path.join(logdir, 'scores.jsonl'),
    ] + glob.glob(os.path.join(logdir, '**', 'metrics.jsonl'), recursive=True) \
      + glob.glob(os.path.join(logdir, '**', 'scores.jsonl'), recursive=True)

    records = []
    for fp in candidates:
        if os.path.exists(fp):
            try:
                with open(fp, 'r') as f:
                    for line in f:
                        line = line.strip()
                        if line:
                            try:
                                records.append(json.loads(line))
                            except json.JSONDecodeError:
                                pass
                if records:
                    print(f"[OK] Loaded {len(records)} records from {fp}")
                    return records
            except Exception as e:
                print(f"[WARN] Could not read {fp}: {e}")

    return records

def extract_metric(records, keys):
    """Extract a metric across records, trying multiple key variants."""
    steps = []
    values = []
    for r in records:
        for k in keys:
            if k in r and r[k] is not None:
                try:
                    s = r.get('step', r.get('agent_step', len(steps)))
                    v = float(r[k])
                    steps.append(int(s))
                    values.append(v)
                    break
                except (ValueError, TypeError):
                    pass
    return np.array(steps), np.array(values)

# Load VLM run records
vlm_records = load_metrics(LOGDIR_VLM)

# Extract metrics
SCORE_KEYS = ['episode/score', 'episode_score', 'score']
LENGTH_KEYS = ['episode/length', 'episode_length', 'length']
DYN_KEYS = ['train/loss/dyn', 'loss/dyn', 'dyn_loss']
IMG_KEYS = ['train/loss/image', 'loss/image', 'image_loss']
VALUE_KEYS = ['train/loss/value', 'loss/value']

vlm_score_s, vlm_score_v = extract_metric(vlm_records, SCORE_KEYS)
vlm_len_s, vlm_len_v = extract_metric(vlm_records, LENGTH_KEYS)
vlm_dyn_s, vlm_dyn_v = extract_metric(vlm_records, DYN_KEYS)
vlm_img_s, vlm_img_v = extract_metric(vlm_records, IMG_KEYS)

print()
print("VLM RUN METRICS:")
print(f"  Total records:    {len(vlm_records)}")
print(f"  Score points:     {len(vlm_score_v)}")
print(f"  Length points:    {len(vlm_len_v)}")
print(f"  Dyn loss points:  {len(vlm_dyn_v)}")
print(f"  Image loss pts:   {len(vlm_img_v)}")

if len(vlm_score_v) > 0:
    print()
    print("  Score stats:")
    print(f"    Episodes:  {len(vlm_score_v)}")
    print(f"    Steps:     {vlm_score_s.min():,} to {vlm_score_s.max():,}")
    print(f"    Min:       {vlm_score_v.min():.1f}")
    print(f"    Max:       {vlm_score_v.max():.1f}")
    print(f"    Mean:      {vlm_score_v.mean():.2f}")
    if len(vlm_score_v) >= 20:
        print(f"    Last 20:   {vlm_score_v[-20:].mean():.2f}")
    if len(vlm_score_v) >= 10:
        print(f"    Last 10:   {vlm_score_v[-10:].mean():.2f}")
else:
    print()
    print("[WARN] No score data found. Plots will use what's available.")

In [ ]:
# Cell 12 — Load VLM call log (JSONL streaming format)
import os, json, numpy as np
from collections import Counter

VLM_LOG_PATH = os.path.join(WORKDIR, 'vlm_calls.jsonl')

vlm_calls = []
if os.path.exists(VLM_LOG_PATH):
    with open(VLM_LOG_PATH, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    vlm_calls.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    print(f"[OK] Loaded {len(vlm_calls)} VLM calls from {VLM_LOG_PATH}")
else:
    print(f"[WARN] No VLM log found at {VLM_LOG_PATH}")

print()
print("VLM CALL ANALYSIS:")
print(f"  Total calls:      {len(vlm_calls)}")

if vlm_calls:
    # Step range
    steps = [c['step'] for c in vlm_calls]
    print(f"  Step range:       {min(steps):,} to {max(steps):,}")

    # Models used
    models = [c.get('model_used', 'unknown') for c in vlm_calls]
    model_counts = Counter(models)
    print(f"  Models used:")
    for m, n in model_counts.most_common():
        print(f"    {m:<35s} {n} calls")

    # Reasons
    reasons = [c.get('reason', 'unknown') for c in vlm_calls]
    reason_counts = Counter(reasons)
    print(f"  Trigger reasons:")
    for r, n in reason_counts.most_common():
        print(f"    {r:<25s} {n} calls")

    # Reward bonuses
    bonuses = [c.get('reward_bonus', 0.0) for c in vlm_calls]
    print(f"  Reward bonuses:")
    print(f"    Mean:      {np.mean(bonuses):+.3f}")
    print(f"    Range:     [{min(bonuses):+.3f}, {max(bonuses):+.3f}]")
    print(f"    Positive:  {sum(1 for b in bonuses if b > 0)} / {len(bonuses)}")
    print(f"    Negative:  {sum(1 for b in bonuses if b < 0)} / {len(bonuses)}")

    # Action distribution
    all_actions = []
    for c in vlm_calls:
        all_actions.extend(c.get('actions', []))
    if all_actions:
        action_counts = Counter(all_actions)
        ACTION_NAMES = {
            0: 'NOOP', 1: 'FIRE', 2: 'UP', 3: 'RIGHT', 4: 'LEFT', 5: 'DOWN',
            6: 'UPRIGHT', 7: 'UPLEFT', 8: 'DOWNRIGHT', 9: 'DOWNLEFT',
            10: 'UPFIRE', 11: 'RIGHTFIRE', 12: 'LEFTFIRE', 13: 'DOWNFIRE',
            14: 'UPRIGHTFIRE', 15: 'UPLEFTFIRE', 16: 'DOWNRIGHTFIRE', 17: 'DOWNLEFTFIRE'
        }
        print(f"  Top actions suggested:")
        for a, n in action_counts.most_common(5):
            name = ACTION_NAMES.get(a, f'action_{a}')
            print(f"    {a:>2} ({name:<15s}) {n} times")

    # Sample descriptions
    print(f"  Sample VLM descriptions:")
    for c in vlm_calls[:3]:
        desc = c.get('description', '')[:80]
        print(f"    Step {c['step']:>7,}: \"{desc}...\"")
else:
    print()
    print("  No VLM calls were made. Possible reasons:")
    print("    - Training stopped before warmup completed (5K steps)")
    print("    - VLM was disabled or all models failed (check Cell 5 output)")
    print("    - Pre-flight check found no working models")

In [ ]:
# Cell 13 — Load baseline data (your existing 787K-step run)
#
# This is loaded from the JSON I generated from your previous notebook.
# The baseline ran 787K steps and got mean score 61.04, max 100.
#
# If you have the file `training_results.json`, place it at:
#   - Kaggle: upload as a dataset, or paste at /kaggle/working/training_results.json
#   - Colab: upload to /content/training_results.json
#
# If the file isn't there, we use embedded fallback data.

import os, json
import numpy as np

# Try to find baseline file
BASELINE_CANDIDATES = [
    os.path.join(WORKDIR, 'training_results.json'),
    '/kaggle/input/training-results/training_results.json',
    '/kaggle/input/baseline/training_results.json',
    '/content/training_results.json',
]

baseline_data = None
for path in BASELINE_CANDIDATES:
    if os.path.exists(path):
        try:
            with open(path, 'r') as f:
                baseline_data = json.load(f)
            print(f"[OK] Loaded baseline from {path}")
            break
        except Exception as e:
            print(f"[WARN] Could not read {path}: {e}")

# Embedded fallback: summary statistics from your previous run
if baseline_data is None:
    print("[INFO] No baseline file found. Using embedded summary from previous 787K-step run.")
    print("[INFO] To get full baseline curve, upload training_results.json to workdir.")

    # Embedded summary from previous run
    baseline_data = {
        'experiment': 'Vanilla DreamerV3 on Boxing (previous run)',
        'total_episodes': 216,
        'total_steps': 787680,
        'score_stats': {
            'min': -11,
            'max': 100,
            'mean': 61.04,
            'first_half_mean': 45.52,
            'second_half_mean': 76.56,
            'last_20_mean': 80.90,
            'last_10_mean': 81.50,
        },
        # Approximation: synthetic episode list matching the statistics
        # Used only if real data isn't loaded
        'episodes': []
    }

# Extract baseline arrays
if baseline_data.get('episodes'):
    base_score_s = np.array([e['step'] for e in baseline_data['episodes']])
    base_score_v = np.array([e['score'] for e in baseline_data['episodes']])
    base_len_v = np.array([e.get('length', 0) for e in baseline_data['episodes']])
    print(f"[OK] Baseline: {len(base_score_v)} episodes, {base_score_s.max():,} steps")
    print(f"     Mean score: {base_score_v.mean():.2f}, Max: {base_score_v.max()}")
else:
    base_score_s = np.array([])
    base_score_v = np.array([])
    base_len_v = np.array([])
    print(f"[INFO] Using summary stats only (no per-episode data)")
    print(f"     Mean: {baseline_data['score_stats']['mean']}, Max: {baseline_data['score_stats']['max']}")

baseline_summary = baseline_data['score_stats']

In [ ]:
# Cell 14 — Plot 1: Learning curves (Baseline vs VLM)
import matplotlib.pyplot as plt
import numpy as np

def smooth(arr, window=10):
    if len(arr) < window:
        return arr, np.arange(len(arr))
    s = np.convolve(arr, np.ones(window)/window, mode='valid')
    idx = np.arange(window-1, len(arr))
    return s, idx

fig, ax = plt.subplots(figsize=(14, 6))

# Baseline
if len(base_score_v) > 0:
    ax.scatter(base_score_s, base_score_v, color='#3498db', alpha=0.15, s=15, label=None)
    s, i = smooth(base_score_v, 15)
    ax.plot(base_score_s[i], s, color='#2980b9', lw=2.5,
            label=f'Baseline (vanilla, n={len(base_score_v)}, max={base_score_v.max():.0f}, mean={base_score_v.mean():.1f})')

# VLM
if len(vlm_score_v) > 0:
    ax.scatter(vlm_score_s, vlm_score_v, color='#e67e22', alpha=0.15, s=15, label=None)
    s, i = smooth(vlm_score_v, 10)
    ax.plot(vlm_score_s[i], s, color='#d35400', lw=2.5,
            label=f'VLM-gated (n={len(vlm_score_v)}, max={vlm_score_v.max():.0f}, mean={vlm_score_v.mean():.1f})')

# Mark VLM call points
if vlm_calls:
    call_steps = [c['step'] for c in vlm_calls]
    # Plot at bottom of chart as ticks
    if len(vlm_score_v) > 0 or len(base_score_v) > 0:
        all_scores = np.concatenate([base_score_v, vlm_score_v]) if len(base_score_v) > 0 else vlm_score_v
        y_min = float(all_scores.min())
        ax.scatter(call_steps, [y_min - 5] * len(call_steps),
                   marker='v', color='red', s=80, alpha=0.7, label=f'VLM calls (n={len(vlm_calls)})', zorder=5)

ax.set_xlabel('Training Steps', fontsize=12)
ax.set_ylabel('Episode Score', fontsize=12)
ax.set_title('Learning Curves: VLM-Gated vs Baseline DreamerV3 on Atari Boxing', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)

plt.tight_layout()
out_path = os.path.join(WORKDIR, 'plot_learning_curves.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"\n[OK] Saved: {out_path}")

In [ ]:
# Cell 15 — Plot 2: VLM call analysis (4 panels)
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('VLM Behavior Analysis', fontsize=14, fontweight='bold')

# Panel 1: VLM calls over time
ax = axes[0, 0]
if vlm_calls:
    steps = [c['step'] for c in vlm_calls]
    surprises = [c.get('surprise', 0) for c in vlm_calls]
    reasons = [c.get('reason', 'unknown') for c in vlm_calls]

    # Color by reason
    colors_map = {'guaranteed': '#3498db', 'adaptive_surprise': '#e74c3c', 'unknown': '#95a5a6'}
    colors = [colors_map.get(r, '#95a5a6') for r in reasons]

    ax.scatter(steps, surprises, c=colors, s=80, alpha=0.7, edgecolor='black', linewidth=0.5)
    ax.set_xlabel('Training Step')
    ax.set_ylabel('Surprise Value')
    ax.set_title(f'VLM Trigger Points (n={len(vlm_calls)})')
    ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
    ax.grid(True, alpha=0.3)

    # Custom legend
    from matplotlib.patches import Patch
    legend_elements = []
    reason_counts = Counter(reasons)
    for r, c in colors_map.items():
        if r in reason_counts:
            legend_elements.append(Patch(facecolor=c, label=f'{r} ({reason_counts[r]})'))
    if legend_elements:
        ax.legend(handles=legend_elements, fontsize=9)
else:
    ax.text(0.5, 0.5, 'No VLM calls', ha='center', va='center', transform=ax.transAxes, fontsize=14, color='gray')
    ax.set_title('VLM Trigger Points')

# Panel 2: Reward bonus distribution
ax = axes[0, 1]
if vlm_calls:
    bonuses = [c.get('reward_bonus', 0) for c in vlm_calls]
    ax.hist(bonuses, bins=15, color='#9b59b6', alpha=0.7, edgecolor='black')
    ax.axvline(x=0, color='red', linestyle='--', alpha=0.7)
    ax.axvline(x=np.mean(bonuses), color='green', linestyle='--', label=f'Mean: {np.mean(bonuses):.3f}')
    ax.set_xlabel('Reward Bonus')
    ax.set_ylabel('Frequency')
    ax.set_title('VLM Reward Bonus Distribution')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
else:
    ax.text(0.5, 0.5, 'No VLM calls', ha='center', va='center', transform=ax.transAxes, fontsize=14, color='gray')
    ax.set_title('VLM Reward Bonus Distribution')

# Panel 3: Action distribution
ax = axes[1, 0]
if vlm_calls:
    all_actions = []
    for c in vlm_calls:
        all_actions.extend(c.get('actions', []))
    if all_actions:
        action_counts = Counter(all_actions)
        labels, counts = zip(*sorted(action_counts.items(), key=lambda x: -x[1])[:8])
        ACTION_NAMES = {
            0: 'NOOP', 1: 'FIRE', 2: 'UP', 3: 'RIGHT', 4: 'LEFT', 5: 'DOWN',
            6: 'UPRT', 7: 'UPLT', 10: 'UPFR', 11: 'RTFR', 12: 'LTFR'
        }
        labels_named = [f"{a}\n{ACTION_NAMES.get(a, '?')}" for a in labels]
        ax.bar(range(len(labels)), counts, color='#1abc9c', alpha=0.8, edgecolor='black')
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels_named, fontsize=9)
        ax.set_xlabel('Action ID')
        ax.set_ylabel('Times Suggested')
        ax.set_title('Top VLM Action Suggestions')
        ax.grid(True, alpha=0.3, axis='y')
    else:
        ax.text(0.5, 0.5, 'No actions', ha='center', va='center', transform=ax.transAxes, fontsize=14, color='gray')
else:
    ax.text(0.5, 0.5, 'No VLM calls', ha='center', va='center', transform=ax.transAxes, fontsize=14, color='gray')
    ax.set_title('Top VLM Action Suggestions')

# Panel 4: Models used
ax = axes[1, 1]
if vlm_calls:
    models = [c.get('model_used', 'unknown') for c in vlm_calls]
    model_counts = Counter(models)
    if model_counts:
        labels, counts = zip(*model_counts.most_common())
        # Truncate long names
        labels_short = [l[:25] + '...' if len(l) > 25 else l for l in labels]
        ax.barh(range(len(labels)), counts, color='#f39c12', alpha=0.8, edgecolor='black')
        ax.set_yticks(range(len(labels)))
        ax.set_yticklabels(labels_short, fontsize=9)
        ax.set_xlabel('Calls')
        ax.set_title('VLM Models Used')
        ax.grid(True, alpha=0.3, axis='x')
        ax.invert_yaxis()
    else:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes, fontsize=14, color='gray')
else:
    ax.text(0.5, 0.5, 'No VLM calls', ha='center', va='center', transform=ax.transAxes, fontsize=14, color='gray')
    ax.set_title('VLM Models Used')

plt.tight_layout()
out_path = os.path.join(WORKDIR, 'plot_vlm_analysis.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"[OK] Saved: {out_path}")

In [ ]:
# Cell 16 — Plot 3: Score distributions and statistical comparison
# scipy is made optional (Kaggle has numpy.strings issue with some scipy versions)
import matplotlib.pyplot as plt
import numpy as np

# Try scipy import - if it fails (numpy.strings issue), we still produce plots, just no stats tests
try:
    from scipy import stats as scipy_stats
    HAS_SCIPY = True
except Exception as e:
    print(f"[INFO] scipy unavailable ({str(e)[:80]}). Will skip stat tests but still plot distributions.")
    HAS_SCIPY = False

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Histograms
ax = axes[0]
if len(base_score_v) > 0:
    ax.hist(base_score_v, bins=20, alpha=0.6, color='#3498db', label=f'Baseline (μ={base_score_v.mean():.1f})', edgecolor='black')
if len(vlm_score_v) > 0:
    ax.hist(vlm_score_v, bins=20, alpha=0.6, color='#e67e22', label=f'VLM (μ={vlm_score_v.mean():.1f})', edgecolor='black')
ax.set_xlabel('Episode Score')
ax.set_ylabel('Frequency')
ax.set_title('Score Distribution Comparison')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Right: Box plot
ax = axes[1]
data_to_plot = []
labels = []
if len(base_score_v) > 0:
    data_to_plot.append(base_score_v)
    labels.append('Baseline')
if len(vlm_score_v) > 0:
    data_to_plot.append(vlm_score_v)
    labels.append('VLM-gated')

if data_to_plot:
    bp = ax.boxplot(data_to_plot, labels=labels, patch_artist=True, showmeans=True)
    colors = ['#3498db', '#e67e22']
    for patch, color in zip(bp['boxes'], colors[:len(bp['boxes'])]):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_ylabel('Episode Score')
    ax.set_title('Score Box Plot')
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
out_path = os.path.join(WORKDIR, 'plot_distributions.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"[OK] Saved: {out_path}")

# Statistical tests (graceful fallback if scipy missing)
print()
print("=" * 60)
print("STATISTICAL COMPARISON")
print("=" * 60)
if len(base_score_v) > 5 and len(vlm_score_v) > 5:
    # Effect size (Cohen's d) - pure numpy
    pooled_std = np.sqrt((base_score_v.var() + vlm_score_v.var()) / 2)
    cohens_d = (vlm_score_v.mean() - base_score_v.mean()) / pooled_std if pooled_std > 0 else 0
    mean_diff = vlm_score_v.mean() - base_score_v.mean()

    print(f"  Mean baseline:       {base_score_v.mean():.2f}")
    print(f"  Mean VLM:            {vlm_score_v.mean():.2f}")
    print(f"  Mean difference:     {mean_diff:+.2f}")
    print(f"  Cohen's d:           {cohens_d:+.3f}")

    if abs(cohens_d) < 0.2:
        es_label = "negligible"
    elif abs(cohens_d) < 0.5:
        es_label = "small"
    elif abs(cohens_d) < 0.8:
        es_label = "medium"
    else:
        es_label = "large"
    print(f"  Effect size:         {es_label}")

    if HAS_SCIPY:
        try:
            t_stat, t_p = scipy_stats.ttest_ind(base_score_v, vlm_score_v, equal_var=False)
            u_stat, u_p = scipy_stats.mannwhitneyu(base_score_v, vlm_score_v, alternative='two-sided')
            print(f"  Welch's t-test:      t={t_stat:.3f}, p={t_p:.4f}")
            print(f"  Mann-Whitney U:      U={u_stat:.0f}, p={u_p:.4f}")
            print(f"  Significant (p<.05): {'YES' if t_p < 0.05 else 'NO'}")
        except Exception as e:
            print(f"  [WARN] scipy tests failed: {e}")
else:
    print("  Not enough data for comparison")
    print(f"    Baseline: {len(base_score_v)} episodes")
    print(f"    VLM:      {len(vlm_score_v)} episodes")


In [ ]:
# Cell 17 — Plot 4: Training loss comparison
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

def smooth(arr, window=10):
    if len(arr) < window:
        return arr, np.arange(len(arr))
    s = np.convolve(arr, np.ones(window)/window, mode='valid')
    idx = np.arange(window-1, len(arr))
    return s, idx

# Dynamics loss
ax = axes[0]
if len(vlm_dyn_v) > 0:
    s, i = smooth(vlm_dyn_v, 15)
    ax.plot(vlm_dyn_s[i], s, color='#e67e22', lw=2, label='VLM-gated')
ax.set_xlabel('Training Steps')
ax.set_ylabel('Dynamics Loss')
ax.set_title('World Model: Dynamics Loss')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
if len(vlm_dyn_v) == 0:
    ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes, fontsize=14, color='gray')

# Image reconstruction loss
ax = axes[1]
if len(vlm_img_v) > 0:
    s, i = smooth(vlm_img_v, 15)
    ax.plot(vlm_img_s[i], s, color='#e67e22', lw=2, label='VLM-gated')
ax.set_xlabel('Training Steps')
ax.set_ylabel('Image Loss')
ax.set_title('World Model: Image Reconstruction Loss')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
if len(vlm_img_v) == 0:
    ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes, fontsize=14, color='gray')

plt.tight_layout()
out_path = os.path.join(WORKDIR, 'plot_losses.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"[OK] Saved: {out_path}")

In [ ]:
# Cell 18 — Final summary report
import os, glob, json
import numpy as np

print("=" * 70)
print("FINAL REPORT — VLM-GATED DREAMERV3 ON ATARI BOXING")
print("=" * 70)
print()

# ============================================================
# Baseline (your previous run)
# ============================================================
print("BASELINE (Vanilla DreamerV3, previous run):")
print(f"  Source:      {baseline_data.get('experiment', 'previous run')}")
print(f"  Episodes:    {baseline_data['total_episodes']}")
print(f"  Steps:       {baseline_data['total_steps']:,}")
print(f"  Score min:   {baseline_summary['min']}")
print(f"  Score max:   {baseline_summary['max']}")
print(f"  Score mean:  {baseline_summary['mean']:.2f}")
print(f"  Last 20 mean: {baseline_summary.get('last_20_mean', 'N/A')}")
print()

# ============================================================
# VLM run
# ============================================================
print("VLM-GATED RUN (this notebook):")
if len(vlm_score_v) > 0:
    print(f"  Episodes:    {len(vlm_score_v)}")
    print(f"  Steps:       {vlm_score_s.max():,}")
    print(f"  Score min:   {vlm_score_v.min():.0f}")
    print(f"  Score max:   {vlm_score_v.max():.0f}")
    print(f"  Score mean:  {vlm_score_v.mean():.2f}")
    if len(vlm_score_v) >= 20:
        print(f"  Last 20 mean: {vlm_score_v[-20:].mean():.2f}")
else:
    print("  No score data (training may have stopped early)")
print()

# ============================================================
# VLM behavior
# ============================================================
print("VLM BEHAVIOR:")
print(f"  Total VLM calls:  {len(vlm_calls)}")
if vlm_calls:
    from collections import Counter

    reasons = Counter(c.get('reason', '?') for c in vlm_calls)
    print(f"  Triggers by reason:")
    for r, n in reasons.most_common():
        print(f"    {r:<20s} {n}")

    models = Counter(c.get('model_used', '?') for c in vlm_calls)
    print(f"  Models used:")
    for m, n in models.most_common():
        print(f"    {m:<35s} {n}")

    bonuses = [c.get('reward_bonus', 0) for c in vlm_calls]
    print(f"  Reward bonuses:")
    print(f"    Mean: {np.mean(bonuses):+.3f}")
    print(f"    Range: [{min(bonuses):+.3f}, {max(bonuses):+.3f}]")
print()

# ============================================================
# Comparison
# ============================================================
if len(vlm_score_v) >= 10:
    print("COMPARISON:")
    delta_mean = vlm_score_v.mean() - baseline_summary['mean']
    delta_max = vlm_score_v.max() - baseline_summary['max']
    print(f"  Mean score Δ:   {delta_mean:+.2f} ({'VLM better' if delta_mean > 0 else 'baseline better' if delta_mean < 0 else 'equal'})")
    print(f"  Max score Δ:    {delta_max:+.2f}")

    if len(vlm_score_v) >= 20:
        last20_vlm = vlm_score_v[-20:].mean()
        last20_base = baseline_summary.get('last_20_mean', 0)
        delta_last20 = last20_vlm - last20_base
        print(f"  Last 20 mean Δ: {delta_last20:+.2f}")

    # Simple verdict
    print()
    if abs(delta_mean) < 5:
        verdict = "INCONCLUSIVE — scores are within noise"
    elif delta_mean > 0:
        verdict = "VLM HELPS — improvement observed"
    else:
        verdict = "VLM HURTS — overhead without benefit"
    print(f"  Verdict: {verdict}")
print()

# ============================================================
# Output files
# ============================================================
print("OUTPUT FILES:")
file_list = []
for pattern in ['plot_*.png', 'vlm_calls.jsonl', 'training_results.json']:
    file_list.extend(glob.glob(os.path.join(WORKDIR, pattern)))

for fp in sorted(set(file_list)):
    sz = os.path.getsize(fp)
    sz_str = f'{sz/1024:.1f} KB' if sz < 1024*1024 else f'{sz/1024/1024:.2f} MB'
    print(f"  {os.path.basename(fp):<35s} {sz_str:>12s}")

# ============================================================
# Export final results JSON
# ============================================================
final_results = {
    'experiment': 'VLM-gated DreamerV3 on Boxing',
    'baseline': {
        'episodes': baseline_data['total_episodes'],
        'steps': baseline_data['total_steps'],
        'score_stats': baseline_summary,
    },
    'vlm_run': {
        'episodes': int(len(vlm_score_v)),
        'steps': int(vlm_score_s.max()) if len(vlm_score_s) > 0 else 0,
        'score_stats': {
            'min': float(vlm_score_v.min()) if len(vlm_score_v) > 0 else None,
            'max': float(vlm_score_v.max()) if len(vlm_score_v) > 0 else None,
            'mean': float(vlm_score_v.mean()) if len(vlm_score_v) > 0 else None,
            'last_20_mean': float(vlm_score_v[-20:].mean()) if len(vlm_score_v) >= 20 else None,
        },
    },
    'vlm_calls': {
        'total': len(vlm_calls),
        'calls': vlm_calls,
    },
}

results_path = os.path.join(WORKDIR, 'final_results.json')
with open(results_path, 'w') as f:
    json.dump(final_results, f, indent=2, default=str)
print(f"\n  {os.path.basename(results_path):<35s} {os.path.getsize(results_path)/1024:.1f} KB (FINAL EXPORT)")

print()
print("=" * 70)
print("DONE — All cells completed. Files ready for report.")
print("=" * 70)